# ATS Evaluation - Task 7: Evaluate Against Human Labels

This notebook compares the AI's ATS scores against a human's own honest judgment, and looks for cases where the AI is clearly wrong - not just an accuracy percentage.

**Important: fill in your OWN scores in the cell below BEFORE running the AI comparison cells**, so your judgment isn't influenced by seeing the AI's numbers first.

In [5]:
import json
from ats_engine_v2 import ats_score

with open("ats_test_data.json", "r") as f:
    data = json.load(f)

pairs = data["pairs"]
print(f"Loaded {len(pairs)} resume/job pairs.")

c:\Users\Mahnoor Sohail\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1956.93it/s]


Loaded 3 resume/job pairs.


## Step 1: Your human scores

Read each resume and job description in `ats_test_data.json` yourself (not the AI's output) and give your own honest score out of 100 for how well each candidate fits the job.

**Replace the `None` values below with YOUR OWN numbers (0-100) before continuing.**

In [4]:
# TODO: Replace None with your own honest score (0-100) for each pair,
# based on YOUR OWN reading of the resume and job description - not the AI's output.
human_scores = {
    "pair_01_strong_match": 95,   # <-- your score here
    "pair_02_medium_match": 80,   # <-- your score here
    "pair_03_weak_match": 55,     # <-- your score here
}

assert all(v is not None for v in human_scores.values()), \
    "Fill in your own scores above before running the rest of the notebook."

## Step 2: Run the AI and compare

In [6]:
comparison_rows = []

for pair in pairs:
    pair_id = pair["pair_id"]
    result = ats_score(pair["resume"], pair["job_description"])

    ai_score_pct = round(result["final_score"] * 100, 1)
    human_score_pct = human_scores[pair_id]
    gap = round(ai_score_pct - human_score_pct, 1)

    comparison_rows.append({
        "pair_id": pair_id,
        "candidate": pair["resume"]["name"],
        "job": pair["job_description"]["title"],
        "human_score": human_score_pct,
        "ai_score": ai_score_pct,
        "gap": gap,
        "sub_scores": result["sub_scores"],
    })

print(f"{'Candidate':<35}{'Human':>8}{'AI':>8}{'Gap':>8}")
for row in comparison_rows:
    print(f"{row['candidate']:<35}{row['human_score']:>7}%{row['ai_score']:>7}%{row['gap']:>7}%")

Candidate                             Human      AI     Gap
Al E. Gator                             95%   80.6%  -14.4%
Albert Gator                            80%   73.0%   -7.0%
Al E. Gator (Agricultural Sciences)     55%   39.3%  -15.7%


## Step 3: Find cases where the AI is clearly wrong

Look at the `gap` column above. For any pair where the gap is large (say, more than ~15 points), dig into `sub_scores` to see WHICH component (skills / experience / semantic / education) caused the AI to disagree with your judgment.

Use the cell below to print the sub-scores for any pair you want to investigate.

In [7]:
# Change this to whichever pair_id you want to investigate
pair_id_to_inspect = "pair_02_medium_match"

row = next(r for r in comparison_rows if r["pair_id"] == pair_id_to_inspect)
print(f"Candidate: {row['candidate']}  ->  {row['job']}")
print(f"Human score: {row['human_score']}%   AI score: {row['ai_score']}%   Gap: {row['gap']}%\n")
print("Sub-scores:")
for component, score in row["sub_scores"].items():
    print(f"  {component}: {round(score * 100, 1)}%")

Candidate: Albert Gator  ->  Marketing Coordinator
Human score: 80%   AI score: 73.0%   Gap: -7.0%

Sub-scores:
  education: 100.0%
  skills: 42.0%
  experience: 100.0%
  semantic: 73.2%


## Step 4: Written error analysis

**Overall pattern:** The AI scored lower than me on all 3 pairs. Small gap on pair_02 (-7), big gaps on pair_01 (-14.4) and pair_03 (-15.7).

**pair_01 (CS → SWE):** Human 95% vs AI 80.6%
- I scored high because of the Google internship — a "prestige" factor the AI can't measure.
- AI's skills (74%) and semantic (71%) were just average, which pulled the total down despite perfect education/experience.
- Fix: not really a bug — just a gap between what a formula can measure and what a human values.

**pair_03 (Ag Sciences → Data Analyst):** Human 55% vs AI 39.3% — biggest gap
- I gave partial credit for real work experience + a relevant degree (trainable candidate).
- AI gave a flat 0% skills score — no concept of "transferable" skills, and skills carries the heaviest weight (35%), so it crushed the total score.
- Fix: add partial credit for adjacent/related experience, or lower the skills weight slightly.

**pair_02 (Marketing → Marketing):** Human 80% vs AI 73% — smallest gap
- AI correctly caught 3 missing required skills I may have underweighted.
- This one looks fair, not a real AI error.